<a href="https://colab.research.google.com/github/goutham3010/Hospital-Readmission-Prediction-System/blob/main/03_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL READMISSION PREDICTION SYSTEM
## NOTEBOOK 03: DATA PREPROCESSING
This notebook handles data cleaning, leakage removal, feature splitting, numerical & categorical pipeline transformation, and preprocessor persistence.


### 1. IMPORT LIBRARIES


In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

print("Libraries imported successfully.")


### 2. LOAD DATASET


In [ ]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

# Auto-detect file location (Google Colab / Local repo)
possible_paths = [
    "/content/hospital_readmission_dataset.csv",
    "hospital_readmission_dataset.csv",
    "../data/raw/hospital_readmission_dataset.csv",
    "data/raw/hospital_readmission_dataset.csv"
]

data_path = next((p for p in possible_paths if os.path.exists(p)), "/content/hospital_readmission_dataset.csv")

df = pd.read_csv(data_path)

print(f"Dataset loaded from: {data_path}")
print("Original shape:", df.shape)
display(df.head())


### 3. CHECK DATA


In [ ]:
# ============================================================
# 3. CHECK DATA
# ============================================================

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate records:")
print(df.duplicated().sum())


### 4. REMOVE DUPLICATES


In [ ]:
# ============================================================
# 4. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates()

print("\nShape after removing duplicates:", df.shape)


### 5. REMOVE UNNECESSARY / RISKY FEATURES


In [ ]:
# ============================================================
# 5. REMOVE UNNECESSARY / RISKY FEATURES
# ============================================================

columns_to_drop = [
    "patient_id",
    "admission_date",
    "readmission_risk_score"
]

# Drop only columns that actually exist
columns_to_drop = [
    col for col in columns_to_drop
    if col in df.columns
]

df = df.drop(columns=columns_to_drop)

print("\nRemoved columns:")
print(columns_to_drop)

print("\nRemaining columns:")
print(df.columns.tolist())


### 6. DEFINE TARGET


In [ ]:
# ============================================================
# 6. DEFINE TARGET
# ============================================================

target = "label"

X = df.drop(columns=[target])
y = df[target]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)


### 7. IDENTIFY FEATURE TYPES


In [ ]:
# ============================================================
# 7. IDENTIFY FEATURE TYPES
# ============================================================

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


### 8. TRAIN / TEST SPLIT


In [ ]:
# ============================================================
# 8. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data :", X_test.shape)


### 9. CHECK TARGET DISTRIBUTION


In [ ]:
# ============================================================
# 9. CHECK TARGET DISTRIBUTION
# ============================================================

print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).round(3))


### 10. NUMERICAL PREPROCESSING


In [ ]:
# ============================================================
# 10. NUMERICAL PREPROCESSING
# ============================================================

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("Numerical pipeline initialized.")


### 11. CATEGORICAL PREPROCESSING


In [ ]:
# ============================================================
# 11. CATEGORICAL PREPROCESSING
# ============================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

print("Categorical pipeline initialized.")


### 12. COMBINE PREPROCESSING


In [ ]:
# ============================================================
# 12. COMBINE PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("Full preprocessor pipeline assembled.")


### 13. FIT PREPROCESSOR ONLY ON TRAINING DATA


In [ ]:
# ============================================================
# 13. FIT PREPROCESSOR ONLY ON TRAINING DATA
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Preprocessor successfully fitted on training data and applied to both sets.")


### 14. CHECK PROCESSED DATA


In [ ]:
# ============================================================
# 14. CHECK PROCESSED DATA
# ============================================================

print("\nProcessed training shape:")
print(X_train_processed.shape)

print("\nProcessed testing shape:")
print(X_test_processed.shape)


### 15. SAVE PREPROCESSOR


In [ ]:
# ============================================================
# 15. SAVE PREPROCESSOR
# ============================================================

save_path = "preprocessor.pkl" if not os.path.exists("/content") else "/content/preprocessor.pkl"

joblib.dump(
    preprocessor,
    save_path
)

# Also save into models folder if present
if os.path.exists("models") or os.path.exists("../models"):
    target_model_dir = "models" if os.path.exists("models") else "../models"
    joblib.dump(preprocessor, os.path.join(target_model_dir, "preprocessor.joblib"))

print(f"\nPreprocessor saved successfully to: {save_path}")


### 16. FINAL CHECK


In [ ]:
# ============================================================
# 16. FINAL CHECK
# ============================================================

print("\n" + "=" * 60)
print("DATA PREPROCESSING COMPLETED")
print("=" * 60)

print("Training samples :", X_train.shape[0])
print("Testing samples  :", X_test.shape[0])
print("Processed train  :", X_train_processed.shape)
print("Processed test   :", X_test_processed.shape)

print("""
Next Step:
04_Feature_Engineering.ipynb / 05_Model_Training.ipynb
""")
